In [14]:
import pandas as pd
import json
import logging

# Setup logging
logging.basicConfig(level=logging.INFO)


In [46]:
# Load the nutrition database from CSV
def load_nutrition_data(csv_file):
    try:
        # Read CSV into pandas DataFrame
        df = pd.read_csv(csv_file)
        
        # Check if the DataFrame is empty
        if df.empty:
            logging.error("The CSV file is empty.")
            return None
        
        # Ensure 'food_name' column exists in the data
        if 'food_name' not in df.columns:
            logging.error("'food_name' column is missing in the CSV.")
            return None
        
        # Log the first few rows of the DataFrame to ensure it's loaded correctly
        logging.debug(f"First few rows of the DataFrame:\n{df.head()}")
        
        # Remove duplicate food names (keep the first occurrence)
        df = df.drop_duplicates(subset='food_name', keep='first')
        
        # Convert the DataFrame into a dictionary indexed by food_name
        nutrition_db = df.set_index('food_name').to_dict(orient='index')
        
        # Convert all food names (keys) to lowercase for case-insensitive matching
        nutrition_db_lower = {key.lower(): value for key, value in nutrition_db.items()}
        
        logging.debug(f"Keys of the nutrition data: {list(nutrition_db_lower.keys())}")  # Log keys
        return nutrition_db_lower
    except Exception as e:
        logging.error(f"Error loading nutrition data: {e}")
        return None


In [47]:
# Household measurement mapping (example)
measurement_mapping = {
    "wet sabzi": 180,  # ~180g per katori for Wet Sabzi
    "dry sabzi": 150,  # ~150g per katori for Dry Sabzi
    "dal": 200,        # ~200g per katori for Dal
    "non-veg curry": 200  # ~200g per katori for Non-Veg Curry
}

# Ingredient quantity conversion (approximate)
conversion_table = {
    "cup": 240,  # 1 cup = 240g
    "teaspoon": 5,  # 1 tsp = 5g
    "tablespoon": 15,  # 1 tbsp = 15g
    "katori": 180  # 1 katori = 180g (specific to Wet Sabzi)
}


In [53]:
def convert_quantity(ingredient, quantity):
    try:
        quantity_parts = quantity.lower().split()
        if len(quantity_parts) < 2:
            logging.error(f"Invalid quantity format for ingredient {ingredient}: {quantity}")
            return 0
        
        amount = float(quantity_parts[0])
        unit = quantity_parts[1]  # Use only the first unit part

        # Optional: normalize plural units
        unit = unit.rstrip('s')  

        if unit in conversion_table:
            return amount * conversion_table[unit]
        else:
            logging.warning(f"Unrecognized unit: {unit} for ingredient {ingredient}")
            return amount
    except Exception as e:
        logging.error(f"Error converting quantity for ingredient {ingredient}: {e}")
        return 0


In [60]:
def calculate_nutrition(ingredients, nutrition_db):
    total_nutrition = {"energy_kcal": 0, "carb_g": 0, "protein_g": 0, "fat_g": 0, "fibre_g": 0}
    
    for item in ingredients:
        ingredient = item["ingredient"].strip().lower()
        quantity = item["quantity"]

        if ingredient not in nutrition_db:
            closest = find_closest_match(ingredient, nutrition_db.keys())
            if closest:
                logging.warning(f"Using closest match for {ingredient}: {closest}")
                ingredient = closest
            else:
                logging.warning(f"Ingredient {ingredient} not found in Nutrition Database.")
                continue

        nutrition_per_serving = nutrition_db[ingredient]
        quantity_in_grams = convert_quantity(ingredient, quantity)

        if quantity_in_grams > 0:
            total_nutrition["energy_kcal"] += (quantity_in_grams / 100) * nutrition_per_serving["unit_serving_energy_kcal"]
            total_nutrition["carb_g"] += (quantity_in_grams / 100) * nutrition_per_serving["unit_serving_carb_g"]
            total_nutrition["protein_g"] += (quantity_in_grams / 100) * nutrition_per_serving["unit_serving_protein_g"]
            total_nutrition["fat_g"] += (quantity_in_grams / 100) * nutrition_per_serving["unit_serving_fat_g"]
            total_nutrition["fibre_g"] += (quantity_in_grams / 100) * nutrition_per_serving["unit_serving_fibre_g"]
    
    return total_nutrition


In [61]:
# Classify dish type based on predefined categories
def classify_dish(dish_name):
    categories = {
        "wet sabzi": ["paneer butter masala", "aloo gobi", "shahi paneer"],
        "dry sabzi": ["dry bhindi", "aloo baingan"],
        "dal": ["dal tadka", "moong dal", "chana dal"],
        "non-veg curry": ["chicken curry", "mutton curry"]
    }

    for category, dishes in categories.items():
        if dish_name.lower() in dishes:
            return category
    return "Unknown"


In [62]:
# Main function to run the pipeline
def process_dish(dish_name, nutrition_db):
    logging.info(f"Processing dish: {dish_name}")

    # Simulate fetching ingredients (replace with real fetching logic)
    ingredients = [
        {"ingredient": "Paneer", "quantity": "0.75 cup cubes"},
        {"ingredient": "Butter", "quantity": "2 teaspoons"},
        {"ingredient": "Tomato", "quantity": "0.5 cup puree"},
        {"ingredient": "Onion", "quantity": "0.5 cup chopped"},
        {"ingredient": "Cream", "quantity": "1 tablespoon"}
    ]

    if not ingredients:
        logging.error(f"No ingredients found for dish: {dish_name}")
        return None

    # Calculate nutrition
    total_nutrition = calculate_nutrition(ingredients, nutrition_db)

    # Classify dish
    dish_type = classify_dish(dish_name)

    # Get standard serving size
    standard_serving = measurement_mapping.get(dish_type, 180)  # Default to 180g if unknown

    # Extrapolate nutrition for a standard serving size
    nutrition_per_serving = {k: v * (standard_serving / 100) for k, v in total_nutrition.items()}

    # Prepare output
    output = {
        "estimated_nutrition_per_200ml_katori": nutrition_per_serving,
        "dish_type": dish_type,
        "ingredients_used": ingredients
    }

    return output


In [63]:
import pandas as pd

# Load the nutrition sheet
nutrition_df = pd.read_excel("Assignment Inputs.xlsx", sheet_name="Nutrition source")

# Strip column names of any leading/trailing spaces
nutrition_df.columns = nutrition_df.columns.str.strip()

# Drop rows where food_name is missing
nutrition_df = nutrition_df.dropna(subset=["food_name"])

# View the top 5 rows
nutrition_df.head()


,food_code,food_name,primarysource,secondarysource,Primary food group,food_group_nin,energy_kj,energy_kcal,carb_g,protein_g,...,unit_serving_folate_ug,unit_serving_vitb1_mg,unit_serving_vitb2_mg,unit_serving_vitb3_mg,unit_serving_vitb5_mg,unit_serving_vitb6_mg,unit_serving_vitb7_ug,unit_serving_vitb9_ug,unit_serving_vitc_mg,unit_serving_carotenoids_ug
0,A001,"Amaranth seed, black (Amaranthus cruentus)",ifct2017,NaN,NaN,Cereals and millets,1490.0,356.11,59.98,14.59,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,A002,"Amaranth seed, pale brown (Amaranthus cruentus)",ifct2017,NaN,NaN,Cereals and millets,1489.0,355.87,61.46,13.27,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,A003,Bajra (Pennisetum typhoideum),ifct2017,NaN,NaN,Cereals and millets,1456.0,347.98,61.78,10.96,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,A004,Barley (Hordeum vulgare),ifct2017,ukfct,NaN,Cereals and millets,1321.0,315.72,61.29,10.94,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,A005,Jowar (Sorghum vulgare),ifct2017,NaN,NaN,Cereals and millets,1398.0,334.12,67.68,9.97,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [64]:
# Example usage in Jupyter Notebook
csv_file = "C:/Users/Tushar Pandey/Assn/Nutrition source.csv"  # Provide the correct path to your CSV file
nutrition_db = load_nutrition_data(csv_file)

# Process the dish
dish_name = "Paneer Butter Masala"
result = process_dish(dish_name, nutrition_db)

# Display the result as formatted JSON
if result:
    print(json.dumps(result, indent=2))


INFO:root:Processing dish: Paneer Butter Masala


{
  "estimated_nutrition_per_200ml_katori": {
    "energy_kcal": NaN,
    "carb_g": NaN,
    "protein_g": NaN,
    "fat_g": NaN,
    "fibre_g": NaN
  },
  "dish_type": "wet sabzi",
  "ingredients_used": [
    {
      "ingredient": "Paneer",
      "quantity": "0.75 cup cubes"
    },
    {
      "ingredient": "Butter",
      "quantity": "2 teaspoons"
    },
    {
      "ingredient": "Tomato",
      "quantity": "0.5 cup puree"
    },
    {
      "ingredient": "Onion",
      "quantity": "0.5 cup chopped"
    },
    {
      "ingredient": "Cream",
      "quantity": "1 tablespoon"
    }
  ]
}
